# CNN Project &mdash; Track A: CNN from scratch (Pierre-Louis)

A hand-built CNN for the Natural Images dataset (8 classes, 128&times;128), in two
categories: **without** data augmentation (`noaug`) and **with** data augmentation (`aug`).

This notebook follows the team's **common experimental protocol** (pistes A, B et C):

1. **Tuning on validation** &mdash; variants are compared on the validation loss, the test set is not evaluated.
2. **Preselection** of the best variant(s) of each category.
3. **Finalist choice with 3 seeds** (42, 43, 44): mean validation loss, simplicity rule, then
   `results/finalistes_A.md` is written **before** reading any test score.
4. **Test measurement** of the finalists, mean &plusmn; std over the seeds.
5. **Pollution study**: finalists replayed, frozen, with 0 / 10 / 20 / 30 % of blurred training images (`src/degradation.py`).
6. Bonus: robustness of the finalists to degraded **test** images (track B protocol).

Every run is saved in its own folder `results/<run_name>/`, never overwritten, and a run that
is already finished is never retrained: the notebook can be stopped and resumed at any time.

## Part 0 &mdash; Data

`data/train`, `data/valid`, `data/test` must already be populated (run the copy cell at
the end of `Data_Exploration.ipynb` once, from the repo root, if you haven't). This
notebook only *reads* them through `src/data_load.load_images`, it never regenerates the
split.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
import os
os.chdir(REPO_ROOT)  # data_load.py and evaluation.py expect to run from the repo root

from src.data_load import IMAGE_SIZE, load_images

for split in ("train", "valid", "test"):
    if not (REPO_ROOT / "data" / split).exists():
        raise FileNotFoundError(
            f"data/{split} is missing -- run the copy cell in Data_Exploration.ipynb first."
        )
print("data/ is ready. Image size:", IMAGE_SIZE)

## Part 1 &mdash; Load the shared split

In [ ]:
X_train, y_train, class_names = load_images("train")
X_val, y_val, _ = load_images("validation")
X_test, y_test, _ = load_images("test")

print("train:", X_train.shape, "val:", X_val.shape, "test:", X_test.shape)
print("classes:", class_names)
print("pixel range:", X_train.min(), "-", X_train.max())

In [ ]:
import numpy as np
import pandas as pd

train_counts = pd.Series(y_train).value_counts().sort_index()
train_counts.index = class_names
train_counts

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, label_idx in zip(axes.ravel(), range(len(class_names))):
    sample = X_train[y_train == label_idx][0]
    ax.imshow(sample)
    ax.set_title(class_names[label_idx])
    ax.axis("off")
plt.tight_layout()
plt.show()

**Observed.** Images come back as raw uint8 arrays (pixels 0-255, not yet
normalized) &mdash; `load_images` only resizes (direct resize by default; `with_padding=True`
is also available and preserves aspect ratio with black borders). Normalization
(`Rescaling(1/255)`) is applied **inside** `model1.build_model`, consistent with the
project's choice to keep `data/` raw so other tracks can apply their own backbone-specific
preprocessing to the same images.

## Part 2 &mdash; Track A: CNN from scratch

The architecture (`src/model1.py`): four `Conv2D -> BatchNorm -> ReLU -> MaxPool`
blocks of increasing width (32/64/128/128), global average pooling instead of flattening to
keep the parameter count down, then a small dense head with dropout. No pretrained weights.
When `augment=True`, an augmentation block (random flip/rotation/zoom/contrast) is inserted
right after the input, active only during training.

The training loop (`src/experiment_a.py`) applies the common rules: Adam, early stopping on
`val_loss` with **patience 3** and best-weights restoration, fixed seed, and a run name built as
`A_cnn4blocks_<noaug|aug>_lr<lr>_do<dropout>_<clean|flou10|...>_seed<seed>`.

In [ ]:
from src.model1 import build_model

build_model(n_classes=len(class_names)).summary()

In [ ]:
from src.experiment_a import (
    MAX_EPOCHS, PATIENCE, POLLUTION_METHOD, POLLUTION_PERCENTS, SEEDS, TUNING_SEED,
    config_key, evaluate_on_test, load_model, run_name, runs_table, train_run, write_finalists,
)

data = {
    "X_train": X_train, "y_train": y_train,
    "X_val": X_val, "y_val": y_val,
    "X_test": X_test, "y_test": y_test,
    "class_names": class_names,
}
print("seeds:", SEEDS, "| max epochs:", MAX_EPOCHS, "| patience:", PATIENCE)

## Part 3 &mdash; Step 1: tuning on the validation set

Each variant is trained once with the tuning seed (42). **The test set is not evaluated**:
the run folders contain the weights, `train_loss.csv`, `valid_loss.csv` and `config.yaml`
(with the validation loss, F1 macro and accuracy), but no `test_output.json`.

Add or remove variants in `TUNING_GRID`; the number of configurations tried per category is
reported in the finalists file, as the protocol requires.

In [ ]:
TUNING_GRID = {
    "noaug": [
        dict(augment=False, lr=1e-3, dropout=0.3),
        dict(augment=False, lr=3e-4, dropout=0.3),
    ],
    "aug": [
        dict(augment=True, lr=1e-3, dropout=0.3),
        dict(augment=True, lr=3e-4, dropout=0.3),
    ],
}

tuning_runs = [
    train_run(data, seed=TUNING_SEED, **cfg)
    for configs in TUNING_GRID.values()
    for cfg in configs
]
tuning_table = runs_table(tuning_runs).sort_values(["category", "val_loss"])
tuning_table[["run", "category", "learning_rate", "dropout_rate", "val_loss", "val_f1_macro",
              "val_accuracy", "epochs_run", "training_seconds"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, category in zip(axes, TUNING_GRID):
    for name in tuning_table.loc[tuning_table["category"] == category, "run"]:
        valid_loss = pd.read_csv(Path("results") / name / "valid_loss.csv")["loss"]
        ax.plot(range(1, len(valid_loss) + 1), valid_loss, "o-", label=name)
    ax.set_title(f"Validation loss - {category}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("val_loss")
    ax.grid(alpha=0.2)
    ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

## Part 4 &mdash; Steps 2 and 3: preselection and finalists (3 seeds)

The `N_PRESELECTED` best variants of each category (lowest validation loss) are retrained
with the 3 seeds. The finalist of each category is the variant with the lowest **mean**
validation loss.

**Simplicity rule**: if two variants are within one standard deviation of each other, keep
the simpler / cheaper one &mdash; override `FINALISTS` by hand in that case. The finalists file
is then written and must be shared with the team **before** running Part 5.

In [ ]:
N_PRESELECTED = 1

val_loss_by_key = tuning_table.set_index("config_key")["val_loss"]
PRESELECTED = {
    category: sorted(configs, key=lambda cfg: val_loss_by_key[config_key(**cfg)])[:N_PRESELECTED]
    for category, configs in TUNING_GRID.items()
}

seed_runs = [
    train_run(data, seed=seed, **cfg)
    for configs in PRESELECTED.values()
    for cfg in configs
    for seed in SEEDS
]
seed_summary = runs_table(seed_runs).groupby(["category", "config_key"]).agg(
    val_loss_mean=("val_loss", "mean"),
    val_loss_std=("val_loss", "std"),
    val_f1_mean=("val_f1_macro", "mean"),
    val_f1_std=("val_f1_macro", "std"),
    n_parameters=("n_parameters", "first"),
    training_seconds_mean=("training_seconds", "mean"),
).sort_values(["category", "val_loss_mean"])
seed_summary

In [ ]:
FINALISTS = {
    category: min(configs, key=lambda cfg: seed_summary.loc[(category, config_key(**cfg)), "val_loss_mean"])
    for category, configs in PRESELECTED.items()
}
n_configs_tried = {category: len(configs) for category, configs in TUNING_GRID.items()}

print(write_finalists(FINALISTS, n_configs_tried).read_text(encoding="utf-8"))

## Part 5 &mdash; Step 4: test measurement of the finalists

Run this part only once `results/finalistes_A.md` has been shared. Each finalist is evaluated
on the test set once per seed with `save_results` (`src/evaluation.py`), which writes
`test_output.json`. Results are reported as mean &plusmn; std over the seeds; the finalists
are not changed afterwards.

In [ ]:
assert (Path("results") / "finalistes_A.md").is_file(), "Write and share the finalists first (Part 4)."

final_runs = {
    category: [run_name(config_key(**cfg), seed) for seed in SEEDS]
    for category, cfg in FINALISTS.items()
}
final_results = {
    category: [evaluate_on_test(data, name) for name in names]
    for category, names in final_runs.items()
}

final_table = runs_table([name for names in final_runs.values() for name in names], with_test=True)
final_summary = final_table.groupby("category").agg(
    finalist=("config_key", "first"),
    test_accuracy_mean=("test_accuracy", "mean"),
    test_accuracy_std=("test_accuracy", "std"),
    test_f1_mean=("test_f1_macro", "mean"),
    test_f1_std=("test_f1_macro", "std"),
    val_loss_mean=("val_loss", "mean"),
    n_parameters=("n_parameters", "first"),
    n_trainable_parameters=("n_trainable_parameters", "first"),
    training_seconds_mean=("training_seconds", "mean"),
)
final_summary["n_configs_tried"] = final_summary.index.map(n_configs_tried)
final_summary

### 5.1 &mdash; Learning curves of the finalists (seed 42)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for category, names in final_runs.items():
    folder = Path("results") / names[0]
    train_loss = pd.read_csv(folder / "train_loss.csv")["loss"]
    valid_loss = pd.read_csv(folder / "valid_loss.csv")["loss"]
    epochs = range(1, len(train_loss) + 1)
    line, = ax.plot(epochs, train_loss, "-", label=f"{category} train")
    ax.plot(epochs, valid_loss, "--", color=line.get_color(), label=f"{category} validation")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

### 5.2 &mdash; Confusion matrices of the finalists (summed over the 3 seeds)

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, len(final_results), figsize=(6 * len(final_results), 5))
for ax, (category, results) in zip(np.atleast_1d(axes), final_results.items()):
    matrix = np.sum([r["confusion_matrix"] for r in results], axis=0)
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f"{category} (3 seeds)")
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
plt.tight_layout()
plt.show()

## Part 6 &mdash; Step 5: pollution study

Each finalist is retrained with its **frozen configuration** while a growing fraction of the
**training** images is blurred with `pollute_train` (`src/degradation.py`), on raw uint8
pixels, before the `Rescaling` done inside the model. Validation and test stay clean, no
hyper-parameter is re-tuned, and the pollution seed is the run seed, so the polluted subsets
are nested (the images blurred at 10 % are also blurred at 20 % and 30 %). Level 0 % is the
clean finalist from Part 5. The test set is evaluated at every run: it is a measurement here.

In [ ]:
pollution_runs = [
    train_run(data, seed=seed, pollution_percent=percent, **cfg)
    for cfg in FINALISTS.values()
    for percent in POLLUTION_PERCENTS
    for seed in SEEDS
]
for name in pollution_runs:
    evaluate_on_test(data, name)

pollution_table = runs_table(pollution_runs, with_test=True)
pollution_summary = pollution_table.groupby(["category", "pollution_percent"]).agg(
    test_f1_mean=("test_f1_macro", "mean"),
    test_f1_std=("test_f1_macro", "std"),
    test_accuracy_mean=("test_accuracy", "mean"),
    test_accuracy_std=("test_accuracy", "std"),
)
pollution_summary.to_csv(Path("results") / "pollution_A.csv")
pollution_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, label in zip(axes, ("test_f1", "test_accuracy"), ("F1 macro (test)", "Accuracy (test)")):
    for category in pollution_summary.index.get_level_values("category").unique():
        subset = pollution_summary.loc[category]
        ax.errorbar(subset.index, 100 * subset[f"{metric}_mean"], yerr=100 * subset[f"{metric}_std"],
                    marker="o", capsize=4, label=f"A {category}")
    ax.set_xlabel(f"Training images polluted with {POLLUTION_METHOD} (%)")
    ax.set_ylabel(f"{label} (%)")
    ax.set_xticks(list(POLLUTION_PERCENTS))
    ax.grid(alpha=0.2)
    ax.legend()
plt.tight_layout()
fig.savefig(Path("results") / "pollution_A.png", dpi=150)
plt.show()

## Part 7 &mdash; Bonus: robustness to degraded test images

Not part of the common protocol: same test-time protocol as track B (Florentin,
`src/robustness_b.py`). The finalists (seed 42, clean) are re-evaluated, without retraining,
on **test** images degraded with `degrade_images` at several levels:

- **flou**: Gaussian blur, radius 0 / 1 / 2 / 3 px
- **pixelisation**: block width 1 / 2 / 4 / 8 px (image shrunk to 128 / 64 / 32 / 16 px, then enlarged back)
- **masque**: black square in the centre, side 0 / 24 / 40 / 64 px

Results go to `results/<run_name>_robustness/` in the same format as track B.

In [ ]:
import json

import yaml

from src.robustness_b import degrade_images

ROBUSTNESS_EXPERIMENTS = [
    ("flou", [0, 1, 2, 3], "Rayon du flou (px)"),
    ("pixelisation", [1, 2, 4, 8], "Largeur d'un bloc (px)"),
    ("masque", [0, 24, 40, 64], "Côté du carré noir (px)"),
]


def evaluate_robustness(name):
    folder = Path("results") / f"{name}_robustness"
    folder.mkdir(parents=True, exist_ok=True)

    model = load_model(name, len(class_names))
    model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    clean_loss, clean_accuracy = model.evaluate(X_test, y_test, verbose=0)
    rows = []
    for method, levels, _ in ROBUSTNESS_EXPERIMENTS:
        for step, level in enumerate(levels):
            applied_level = IMAGE_SIZE // level if method == "pixelisation" else level
            if step == 0:
                loss, accuracy = clean_loss, clean_accuracy
            else:
                altered = degrade_images(X_test, method, applied_level)
                loss, accuracy = model.evaluate(altered, y_test, verbose=0)
                del altered
            rows.append({
                "method": method,
                "level": level,
                "intermediate_size_px": applied_level if method == "pixelisation" else None,
                "accuracy": float(accuracy),
                "loss": float(loss),
                "accuracy_drop_pp": float(100 * (clean_accuracy - accuracy)),
            })
            print(f"{name}  {method:13} {level:2}px | accuracy {accuracy:.2%} | loss {loss:.4f}")

    metrics = pd.DataFrame(rows)
    metrics.to_csv(folder / "robustness_metrics.csv", index=False)
    with (folder / "test_output.json").open("w", encoding="utf-8") as file:
        json.dump({"n_images": len(y_test), "conditions": rows}, file, indent=4)
    with (folder / "config.yaml").open("w", encoding="utf-8") as file:
        yaml.safe_dump({
            "weights": (Path("results") / name / "model.weights.h5").as_posix(),
            "resize": "direct",
            "image_size": IMAGE_SIZE,
            "split": "test",
            "blur_radius_px": ROBUSTNESS_EXPERIMENTS[0][1],
            "pixel_block_width_px": ROBUSTNESS_EXPERIMENTS[1][1],
            "black_square_side_px": ROBUSTNESS_EXPERIMENTS[2][1],
        }, file)
    return metrics


robustness = {category: evaluate_robustness(names[0]) for category, names in final_runs.items()}

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (method, levels, axis_label) in zip(axes, ROBUSTNESS_EXPERIMENTS):
    for category, metrics in robustness.items():
        subset = metrics[metrics["method"] == method]
        ax.plot(subset["level"], 100 * subset["accuracy"], "o-", label=f"A {category}")
    ax.set_title(method.capitalize())
    ax.set_xlabel(axis_label)
    ax.set_ylabel("Accuracy (%)")
    ax.set_xticks(levels)
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.2)
    ax.legend()
plt.tight_layout()
for names in final_runs.values():
    fig.savefig(Path("results") / f"{names[0]}_robustness" / "robustness_curves.png", dpi=150)
plt.show()

In [ ]:
pd.concat([
    metrics.assign(category=category) for category, metrics in robustness.items()
]).pivot_table(index=["method", "level"], columns="category", values="accuracy_drop_pp").round(1)

## Summary

All numbers below come from `results/` (validation for every choice, test only for the finalists and the pollution study).

**Model.** A CNN trained from scratch: 4 blocks `Conv2D-BatchNorm-ReLU-MaxPool` (32, 64, 128, 128 filters), global average pooling, `Dense(128)`, dropout and a softmax over the 8 classes. 259,784 parameters (259,080 trainable). Pixels are rescaled to [0, 1] inside the model.

**Tuning (Part 3).** 2 configurations per category, seed 42, selected on the validation loss:

| Category | lr 1e-3 | lr 3e-4 |
|---|---|---|
| noaug | 1.080 | **0.632** |
| aug | 0.851 | **0.559** |

A learning rate of 3e-4 wins clearly in both categories; dropout stays at 0.3. The tuning budget is small (2 configurations per category) and is reported as such in `results/finalistes_A.md`.

**Finalists (Part 4).** Mean &plusmn; std over seeds 42, 43, 44, validation only:

| Finalist | val_loss | val F1 macro |
|---|---|---|
| `A_cnn4blocks_noaug_lr0.0003_do0.3` | 0.478 &plusmn; 0.165 | 0.808 &plusmn; 0.069 |
| `A_cnn4blocks_aug_lr0.0003_do0.3` | 0.509 &plusmn; 0.046 | 0.809 &plusmn; 0.014 |

The gap between the two means (0.03) is far below the standard deviation: the two categories are tied on the validation loss. Both are kept as finalists, one per category, as required by the protocol.

**Test (Part 5).** Mean &plusmn; std over the 3 seeds (`results/test_A.csv`):

| Finalist | Test accuracy | Test F1 macro | Training time (clean run) |
|---|---|---|---|
| noaug | 83.2 % &plusmn; 7.4 | 0.814 &plusmn; 0.076 | about 15 min |
| aug | 82.3 % &plusmn; 3.0 | 0.807 &plusmn; 0.028 | about 13 min |

- Same picture as on the validation: no difference on the mean, but data augmentation makes training **much more stable** across seeds (std divided by about 2.5 on the test, 3.6 on the validation loss). Without augmentation the result depends heavily on the seed (75.7 % to 90.6 % accuracy).
- Training times are on CPU; some runs were stretched by the computer going to sleep, so the mean stored in `test_A.csv` for `aug` (31 min) is not representative.
- **Confusions** (matrices summed over the 3 seeds): the main error is **cat &harr; dog**. Without augmentation, 168 of the 399 cat test images (133 per seed) are predicted as dog, and many images of several classes fall into *flower*; with augmentation the cat/dog confusion remains (103 dogs predicted as cat) but the errors are spread more evenly. *Fruit*, *person* and *motorbike* are almost always correct. Cats and dogs share textures, colours and poses, which a small network trained on 4,829 images cannot fully separate.

**Pollution (Part 6).** Finalists retrained with a growing share of the training images blurred (Gaussian blur, radius 1 px), F1 macro on the test, mean &plusmn; std over 3 seeds (`results/pollution_A.csv`, `results/pollution_A.png`):

| Blurred training images | noaug | aug |
|---|---|---|
| 0 % | 0.814 &plusmn; 0.076 | 0.807 &plusmn; 0.028 |
| 10 % | 0.855 &plusmn; 0.038 | 0.803 &plusmn; 0.015 |
| 20 % | 0.826 &plusmn; 0.059 | 0.776 &plusmn; 0.023 |
| 30 % | 0.849 &plusmn; 0.025 | 0.824 &plusmn; 0.021 |

The curves are **flat**: up to 30 % of blurred training images, no level is distinguishable from the clean run given the seed variance. A 1 px blur on part of the training set acts more like an extra augmentation than like a corruption. Track B shows the same flat curve, so this level of pollution is too weak to separate the tracks; a stronger setting (larger radius or higher share) is a point to agree on as a team, as planned in the protocol.

**Bonus: robustness to degraded test images (Part 7).** Seed 42 finalists, no retraining, test accuracy:

| Degradation | noaug | aug |
|---|---|---|
| None | 75.8 % | 78.8 % |
| Blur 1 px / 3 px | 52.7 % / 47.0 % | 60.1 % / 47.3 % |
| Pixelation, 2 px / 8 px blocks | 44.4 % / 46.0 % | 55.6 % / 41.6 % |
| Black square 40 px / 64 px | 69.2 % / 62.4 % | 78.3 % / 68.4 % |

- The from-scratch CNN relies heavily on fine textures: blurring or pixelating **every** test image costs 20 to 35 points, even at the lightest level. This contrasts with the pollution study, where blurring part of the training set had no effect.
- Augmentation helps against occlusion: with a 40 px black square in the centre, the augmented model loses nothing, the other one loses 7 points.

**Takeaways for track A.**
- A small CNN trained from scratch reaches about 82-83 % test accuracy, with a large seed-to-seed variability.
- Data augmentation does not raise the mean score but makes training more reliable and more robust to occlusion.
- The main confusion is cat &harr; dog.
- These are the reference points for tracks B and C (transfer learning and fine-tuning).